# Benchmark of ODE Solvers Using Implicit Kaps' Problem

## Problem

### Equation

$$
\begin{aligned}
    \varepsilon \dot{y_1} & = -(1 + 2 \varepsilon) y_1 + y_2^2 \\
    \dot{y_2} & = y_1 - y_2 - y_2^2
\end{aligned}
$$

### Initial Condition

$$
\begin{aligned}
    y_1(0) & = 1 \\
    y_2(0) & = 1
\end{aligned}
$$

### Notes

- This problem is stiff when $\varepsilon$ is small.
- This problem becomes DAE when $\varepsilon = 0$. For this case, only solvers supporting DAEs are applied.


## Results

In [ ]:
import num_collect_docs_utils.plot_common

num_collect_docs_utils.plot_common.set_common_configuration()

In [ ]:
# Load data
import gzip
import msgpack
import pandas

epsilon_list = [
    # (epsilon in file name, label in plot)
    ("0", "1"),
    ("3", "1e-3"),
    ("6", "1e-6"),
    ("inf", "0"),
]

data_frames_per_epsilon = []
for epsilon, label in epsilon_list:
    with gzip.open(f"data/implicit_kaps_problem{epsilon}.data", mode="rb") as file:
        results = msgpack.unpack(file)
    data_frames_per_epsilon.append(
        pandas.DataFrame(
            {
                "Epsilon": [label] * len(results["solver_list"]),
                "Solver": results["solver_list"],
                "Err. Tol.": results["tolerance_list"],
                "Error Rate": results["error_rate_list"],
                "Time [sec]": results["time_list"],
            }
        )
    )

data_frame = pandas.concat(data_frames_per_epsilon, ignore_index=True)

In [ ]:
# Show plots
import plotly.express
import num_collect_docs_utils.plot_common

figure = plotly.express.line(
    data_frame,
    x="Time [sec]",
    y="Error Rate",
    color="Solver",
    line_dash="Solver",
    facet_row="Epsilon",
    hover_data=["Solver", "Err. Tol.", "Error Rate", "Time [sec]"],
    markers=True,
    log_x=True,
    log_y=True,
    title="Work-Error Diagram of ODE solvers for Implicit Kaps' Problem",
)
figure.update_layout(height=1400)
num_collect_docs_utils.plot_common.show_figure(figure)

## Execution Environment

- CPU: Intel(R) Core(TM) Ultra 5 125H (14 cores, 18 processors)
- Memory: 32 GB
- Commit: `0a3348313f8a09eacb9713bd61e587e25deb42fe`
- Command: `./build/Release/bin/bench_ode_implicit_kaps_problem results/`
